In [23]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

In [25]:


candidates = [
    Path("assets/pdf/EMPLOYEE-HANDBOOK.pdf"),
    Path("../assets/pdf/EMPLOYEE-HANDBOOK.pdf"),
    Path("projects/01-langchain/assets/pdf/EMPLOYEE-HANDBOOK.pdf"),
]

pdf_path = next((p for p in candidates if p.exists()), None)
if pdf_path is None:
    raise FileNotFoundError("Could not find EMPLOYEE-HANDBOOK.pdf in expected paths")

loader = PyPDFLoader(str(pdf_path.resolve()))
data = loader.load()

print(data)

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-02-01T13:42:37-05:00', 'author': 'gretac', 'moddate': '2024-02-01T13:42:37-05:00', 'source': '/Users/angelo/projects/genai-portfolio/projects/01-langchain/assets/pdf/EMPLOYEE-HANDBOOK.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1'}, page_content='ACME Plumbing Durham, Inc \nEMPLOYEE HANDBOOK UPDATED 1 February 2024 \n 1 \n \n \n \n                         \n \n \n \n                                \n \n \n \n \n4625 Industry Ln \n \nDurham, NC  27713 \n \n919-688-1348 \n \nwww.acmeplumbing.com'), Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-02-01T13:42:37-05:00', 'author': 'gretac', 'moddate': '2024-02-01T13:42:37-05:00', 'source': '/Users/angelo/projects/genai-portfolio/projects/01-langchain/assets/pdf/EMPLOYEE-HANDBOOK.pdf', 'total_pages': 48, 'page

In [26]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, 
    chunk_overlap = 200, 
    add_start_index = True,
)

all_splits = text_splitter.split_documents(data)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = InMemoryVectorStore(embeddings)

ids = vector_store.add_documents(documents=all_splits)

In [28]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

page_content='addition to their wages for that day's work.    
 
VACATION: Employees will become eligible for paid vacation after completion of 1 
year of satisfactory employment as follows: 
• After one (1) year of service, you will have one (1) week (40 hours) of 
vacation. 
• After three (3) years of service, you will have two (2) weeks. 
• Beginning with the 6th anniversary, you will receive two (2) weeks plus an 
additional day (8 hours) for each year of service up to three (3) weeks total. 
(It takes ten (10) years to get there but you will have three (3) weeks of paid 
vacation plus five (5) paid holidays per year. 
 
Vacation must be taken at the convenience of the Company and should be requested 
at least 30 days in advance. ACME Plumbing Co., Inc. will make every effort to 
schedule work in such a manner that all employees will be able to use their vacation 
when requested, but this may not always be possible.' metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creat

In [29]:
# RAG Agent

In [30]:
@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content
    

In [31]:
SYSTEM_PROMPT = "You are a helpful agent that can search the employee handbook for information"

agent = create_agent(
    model="gpt-5-nano",
    tools=[search_handbook],
    system_prompt=SYSTEM_PROMPT,
)

In [32]:
response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in the first year?")]}
)

print(response)

{'messages': [HumanMessage(content='How many days of vacation does an employee get in the first year?', additional_kwargs={}, response_metadata={}, id='6ae37782-1b83-4e58-8114-6f79ac926e8e'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 157, 'total_tokens': 313, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DGv3mbb1ut8urqgyXr9uuuxUa9q9M', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cca8f-f40a-7e03-baf5-5569e9765418-0', tool_calls=[{'name': 'search_handbook', 'args': {'query': 'vacation days first year'}, 'id': 'call_RGj5rrukKpJAtRzPTa55qjSG', 'type': 'tool_call'}], invalid_tool_ca